# Camada Silver: Tratamento, Padronização e Qualidade de Dados

Processamento das tabelas da camada Bronze para consolidação da camada Silver do modelo medalhão. Esta etapa executa a limpeza, remoção de registros duplicados, padronização textual, conversão de tipos (datas, valores numéricos e carimbos de tempo) e o cruzamento relacional entre as fontes operacionais da V-Credit.

### Diretrizes Arquiteturais e Governança de Dados
- **Idempotência:** Persistência em modo `overwrite` para assegurar reprodutibilidade e evitar duplicações em execuções concorrentes pela equipe.
- **Auditoria e Isolamento:** A coluna técnica `ingestion_timestamp` permanece restrita à camada Bronze. O histórico de qualidade é auditado via `dq_log` (modo `append`, segregado por `run_id`).
- **Fonte da Verdade Temporal:** Os registros de horário consolidados em `Chamados_Hora.CSV` constituem a única fonte da verdade para o cálculo de durações (`*_calculado_segundos`), enquanto os tempos reportados em `chamados.csv` (`*_informado_segundos`) são mantidos exclusivamente para fins de auditoria comparativa.
- **Contrato de Validade (Flags `fl_*`):** A camada Silver define as regras de integridade e validade consumidas pela camada Gold.
- **Estratégia de Validação (DQ):** Lacunas estruturais previstas no case são sinalizadas como diagnósticos informativos (`WARNING`), e perdas volumétricas de ingestão são registradas como `INFO`. Validações críticas de unicidade e integridade referencial operam como bloqueantes (`BLOCKING`), interrompendo o pipeline em caso de falha para garantir a consistência analítica antes da persistência na camada Gold.

In [0]:
%run ./00_Config

In [0]:
# Importação das bibliotecas utilitárias do Python e PySpark
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F

# Identificador desta execução, para separar as execuções no histórico do dq_log
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Inicialização da lista global para rastreamento de todas as métricas de qualidade (DQ)
dq_results = []

def register_dq(table_name: str, check_name: str, severity: str, total: int, affected: int) -> None:
    """
    Registra o resultado de uma validação na lista global (gravada em silver.dq_log).
    severity: BLOCKING (interrompe o pipeline), WARNING (diagnóstico) ou INFO (apenas informativo).
    affected_rows: linhas (ou chaves) que falharam na validação; nas linhas INFO, o volume descartado.
    """
    dq_results.append(
        Row(
            run_id=run_id,
            table_name=table_name,
            check_name=check_name,
            severity=severity,
            total_rows=total,
            affected_rows=affected,
            passed=affected == 0 or severity == "INFO",
            checked_at=datetime.now(),
        )
    )


def dq_check(table_name: str, check_name: str, df, condition) -> None:
    """
    Executa uma validação crítica (bloqueante) de qualidade de dados.
    Caso existam linhas que falhem na condição, registra o erro; o notebook é interrompido antes de gravar a Silver.
    """
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    register_dq(table_name, check_name, "BLOCKING", total, failed)

    status = "PASS" if failed == 0 else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas com falha")


def dq_warning(table_name: str, check_name: str, df, condition) -> None:
    """
    Executa uma validação informativa (não bloqueante) para diagnóstico de lacunas.
    Registra o aviso no log de auditoria sem interromper o fluxo do pipeline.
    """
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    register_dq(table_name, check_name, "WARNING", total, failed)

    status = "PASS" if failed == 0 else "WARN"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} registros identificados com lacuna (informativo)")


def dq_check_unique(table_name: str, check_name: str, df, key_cols: list) -> None:
    """
    Valida a unicidade de chaves primárias (simples ou compostas) na tabela.
    Garante que não há duplicidade de registros para os identificadores unívocos.
    """
    total = df.count()
    duplicates = df.groupBy(*key_cols).count().filter(F.col("count") > 1).count()
    register_dq(table_name, check_name, "BLOCKING", total, duplicates)

    status = "PASS" if duplicates == 0 else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas encontradas")

## 1. Tratamento da Dimensão de Atendentes

Validação de unicidade por identificador de atendente, conversão de datas de admissão e desligamento, e padronização textual de status e níveis de atendimento (harmonizando variações como `1`, `Nível 1` e `N2` para o formato numérico padrão). Ausências na data de desligamento são preservadas para atendentes com vínculo ativo.

In [0]:
# Leitura da tabela bronze de atendentes
df_bronze_atendentes = spark.table(f"{bronze_schema}.tb_base_atendentes").drop("ingestion_timestamp")

# Tratamento, padronização e tipagem da base de atendentes
df_silver_atendentes = (
    df_bronze_atendentes
    .filter(F.col("id_atendente").isNotNull())
    .withColumn("id_atendente", F.trim(F.col("id_atendente")))
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    .withColumn("nivel_atendimento", F.regexp_extract(F.col("nivel_atendimento"), r"(\d)", 1).try_cast("int"))
    .withColumn("data_admissao", F.try_to_date(F.col("data_admissao"), "dd/MM/yyyy"))
    .withColumn("data_desligamento", F.try_to_date(F.col("data_desligamento"), "dd/MM/yyyy"))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
)

# Execução das validações de qualidade de dados
dq_check_unique("silver.tb_atendentes", "id_atendente único", df_silver_atendentes, ["id_atendente"])
dq_check("silver.tb_atendentes", "nível de atendimento válido", df_silver_atendentes, F.col("nivel_atendimento").isin(1, 2))

display(df_silver_atendentes.limit(10))

## 2. Tratamento das Dimensões de Apoio (Motivos, Canais e Clientes)

Processamento e padronização das dimensões relacionais de suporte. As chaves primárias são auditadas quanto à unicidade sem descarte arbitrário de registros.
- **Motivos:** Harmonização da nomenclatura e mapeamento das categorias em conformidade com os cinco eixos analíticos definidos no case da V-Credit.
- **Canais:** Padronização de nomes, tipos e status operacionais.
- **Clientes:** Tratamento de múltiplos formatos de data de cadastro (`dd/MM/yyyy` e `yyyy-MM-dd`). Inconsistências ou lacunas em idades são mantidas e monitoradas via diagnóstico de qualidade, preservando a integridade do dado histórico original.

In [0]:
# Leitura das tabelas de origem na camada Bronze
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_base_motivos").drop("ingestion_timestamp")
df_bronze_canais = spark.table(f"{bronze_schema}.tb_canais").drop("ingestion_timestamp")
df_bronze_clientes = spark.table(f"{bronze_schema}.tb_clientes").drop("ingestion_timestamp")

# Mapeamento auxiliar de categorias por id_motivo, para motivos sem categoria preenchida
# Cadastral: "Transferência de agência" (14) e "Cancelamento de conta" (16)
# Cartão: "Contratação de cartão adicional" (12)
mapa_categoria_por_id_motivo = F.create_map([
    F.lit("14"), F.lit("Cadastral"),
    F.lit("16"), F.lit("Cadastral"),
    F.lit("12"), F.lit("Cartão")
])

# Categorias oficiais definidas no case
categorias_case = ["Financeiro", "Cartão", "Cadastral", "Atendimento", "Benefícios"]

# Processamento da dimensão de motivos
df_silver_motivos = (
    df_bronze_motivos
    .filter(F.col("id_motivo").isNotNull())
    .withColumn("id_motivo", F.trim(F.col("id_motivo")))
    .withColumn("nome_motivo", F.initcap(F.lower(F.trim(F.col("nome_motivo")))))
    .withColumn("categoria_original", F.col("categoria"))
    .withColumn("categoria",
        F.when(F.trim(F.col("categoria")) == "", None)
        .otherwise(F.trim(F.col("categoria")))
    )
    .withColumn("categoria", F.coalesce(F.col("categoria"), mapa_categoria_por_id_motivo[F.col("id_motivo")]))
    .withColumn("categoria", F.initcap(F.lower(F.trim(F.col("categoria")))))
    .withColumn("categoria",
        F.when(F.col("categoria") == "Cartao", "Cartão")
        .when(F.col("categoria") == "Beneficios", "Benefícios")
        .otherwise(F.col("categoria"))
    )
    .withColumn("criticidade", F.lower(F.trim(F.col("criticidade"))))
)

# Processamento da dimensão de canais
df_silver_canais = (
    df_bronze_canais
    .filter(F.col("id_canal").isNotNull())
    .withColumn("id_canal", F.trim(F.col("id_canal")))
    .withColumn("nome_canal", F.initcap(F.lower(F.trim(F.col("nome_canal")))))
    .withColumn("tipo_canal", F.initcap(F.lower(F.trim(F.col("tipo_canal")))))
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
)

# Processamento da dimensão de clientes com tratamento de múltiplos formatos de data
df_silver_clientes = (
    df_bronze_clientes
    .filter(F.col("id_cliente").isNotNull())
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("regiao", F.initcap(F.lower(F.trim(F.col("regiao")))))
    .withColumn("idade", F.col("idade").try_cast("int"))
    .withColumn("data_cadastro",
        F.coalesce(
            F.try_to_date(F.col("data_cadastro"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_cadastro"), "yyyy-MM-dd"),
        )
    )
    .withColumn("dia_vencimento_fatura", F.col("dia_vencimento_fatura").try_cast("int"))
)

categorias_validas = ["Financeiro", "Cartão", "Cadastral", "Atendimento", "Benefícios"]
qtd_motivos_sem_categoria_valida = df_silver_motivos.filter(
    F.col("categoria").isNull() | ~F.col("categoria").isin(categorias_validas)
).count()

assert qtd_motivos_sem_categoria_valida == 0, (
    f"Silver inválida: {qtd_motivos_sem_categoria_valida} motivos sem categoria padronizada."
)

# Validações de qualidade de dados para as dimensões de apoio
dq_check_unique("silver.tb_motivos", "id_motivo único", df_silver_motivos, ["id_motivo"])
dq_check_unique("silver.tb_canais", "id_canal único", df_silver_canais, ["id_canal"])
dq_check_unique("silver.tb_clientes", "id_cliente único", df_silver_clientes, ["id_cliente"])
dq_warning("silver.tb_motivos", "categoria entre as cinco do case", df_silver_motivos, F.col("categoria").isin(*categorias_case))
dq_warning("silver.tb_clientes", "data de cadastro convertida", df_silver_clientes, F.col("data_cadastro").isNotNull())
dq_warning("silver.tb_clientes", "idade preenchida (lacuna operacional legada)", df_silver_clientes, F.col("idade").isNotNull() & (F.col("idade") > 0))

display(df_silver_motivos)
display(df_silver_canais)
display(df_silver_clientes.limit(10))

## 3. Processamento Temporal de Chamados

Tratamento da volumetria de horários proveniente de `Chamados_Hora.CSV` para suprir lacunas e inconsistências nos carimbos operacionais da base principal.
- **Cálculo de Durações:** Derivação temporal em segundos para tempo de espera (`hora_inicio - hora_abertura`) e tempo de atendimento (`hora_finalizacao - hora_inicio`).
- **Flags de Validade (`fl_*`):** Atribuição de indicadores booleanos de integridade que determinam se as durações calculadas são válidas e não negativas, servindo como filtro restritivo para agregações analíticas na camada Gold. Lacunas operacionais previstas são mantidas e sinalizadas para rastreabilidade.

In [0]:
# Leitura da tabela bronze de horários de chamados
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.tb_chamados_hora").drop("ingestion_timestamp")

# Padronização, conversão de timestamps e cálculo de durações
formato_hora = "dd/MM/yyyy 'às' HH:mm:ss"

df_silver_chamados_hora = (
    df_bronze_chamados_hora
    .filter(F.col("ID_Chamado").isNotNull())
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente_horario")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento")
    .withColumn("id_chamado", F.trim(F.col("id_chamado")))
    .withColumn("id_cliente_horario", F.trim(F.col("id_cliente_horario")))
    .withColumn("hora_abertura_chamado", F.try_to_timestamp(F.col("hora_abertura_chamado"), F.lit(formato_hora)))
    .withColumn("hora_inicio_atendimento", F.try_to_timestamp(F.col("hora_inicio_atendimento"), F.lit(formato_hora)))
    .withColumn("hora_finalizacao_atendimento", F.try_to_timestamp(F.col("hora_finalizacao_atendimento"), F.lit(formato_hora)))
    .withColumn("tempo_espera_calculado_segundos",
        F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("tempo_atendimento_calculado_segundos",
        F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
    # Flags de validade (contrato com a Gold): tempo calculado, não nulo e não negativo
    .withColumn("fl_espera_valida", F.coalesce(F.col("tempo_espera_calculado_segundos") >= 0, F.lit(False)))
    .withColumn("fl_atendimento_valido", F.coalesce(F.col("tempo_atendimento_calculado_segundos") >= 0, F.lit(False)))
    .withColumn("fl_ciclo_atendimento_valido", F.col("fl_espera_valida") & F.col("fl_atendimento_valido"))
)

# Validações de integridade temporal
dq_check_unique("silver.tb_chamados_hora", "id_chamado único", df_silver_chamados_hora, ["id_chamado"])
dq_warning("silver.tb_chamados_hora", "horário de abertura convertido", df_silver_chamados_hora, F.col("hora_abertura_chamado").isNotNull())
dq_warning("silver.tb_chamados_hora", "horário de início preenchido", df_silver_chamados_hora, F.col("hora_inicio_atendimento").isNotNull())
dq_warning("silver.tb_chamados_hora", "horário de finalização convertido", df_silver_chamados_hora, F.col("hora_finalizacao_atendimento").isNotNull())
dq_warning("silver.tb_chamados_hora", "tempo de espera não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_espera_calculado_segundos") >= 0, F.lit(True)))
dq_warning("silver.tb_chamados_hora", "tempo de atendimento não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_atendimento_calculado_segundos") >= 0, F.lit(True)))

display(df_silver_chamados_hora.limit(10))

## 4. Tratamento da Tabela Fato de Chamados

Consolidação da tabela transacional principal da operação.
- **Limpeza e Deduplicação:** Padronização textual de canais, motivos e status de resolução, seguida de remoção de duplicatas exatas.
- **Rastreabilidade Temporal:** Manutenção segregada de métricas informadas (`*_informado_segundos`) e calculadas (`*_calculado_segundos`), evidenciando divergências operacionais frente à fonte oficial da verdade.
- **Validação Cruzada:** Integração com as dimensões de apoio e auditoria de integridade referencial para verificação de existência de clientes, atendentes e consistência de vínculos cadastrais.

In [0]:
# Leitura da base bronze de chamados
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados").drop("ingestion_timestamp")

# Limpeza, normalização de texto (incluindo motivo_chave sem acentos) e remoção de duplicados
df_silver_chamados = (
    df_bronze_chamados
    .filter(F.col("id_chamado").isNotNull())
    .withColumn("id_chamado", F.trim(F.col("id_chamado")))
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
    .withColumn("id_atendente", F.trim(F.col("id_atendente")))
    .withColumn("motivo", F.lower(F.trim(F.col("motivo"))))
    .withColumn("motivo_chave", F.translate(F.col("motivo"), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc"))
    .withColumn("canal", F.lower(F.trim(F.col("canal"))))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"[._]+", " "))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"\s+", " "))
    .withColumn("canal",
        F.when(F.col("canal").isin("u r a", "ura"), "ura")
        .when(F.col("canal").isin("chat bot", "chatbot"), "chatbot")
        .when(F.col("canal").isin("atend inicial", "atendimento inicial"), "atendimento inicial")
        .when(F.col("canal").isin("atend especializado", "atendimento especializado"), "atendimento especializado")
        .otherwise(F.col("canal"))
    )
    .withColumn("resolvido", F.lower(F.trim(F.col("resolvido"))))
    .withColumn("resolvido",
        F.when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
        .when(F.col("resolvido").isin("não", "nao", "n", "0", "nao resolvido"), "Não")
        .otherwise("Não informado")
    )
    .withColumn("tempo_espera", F.when(F.upper(F.col("tempo_espera")) == "NULL", None).otherwise(F.col("tempo_espera")).try_cast("long"))
    .withColumn("tempo_atendimento", F.when(F.upper(F.col("tempo_atendimento")) == "NULL", None).otherwise(F.col("tempo_atendimento")).try_cast("long"))
    .withColumnRenamed("tempo_espera", "tempo_espera_informado_segundos")
    .withColumnRenamed("tempo_atendimento", "tempo_atendimento_informado_segundos")
    .drop("hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento")
    .dropDuplicates()
)

# Criação de referências para cruzamento usando a chave sem acentos para garantir o join
df_motivos_join = df_silver_motivos.withColumn("motivo_chave", F.translate(F.lower(F.trim(F.col("nome_motivo"))), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc"))
df_canais_join = df_silver_canais.withColumn("canal", F.lower(F.trim(F.col("nome_canal"))))
df_clientes_referencia = df_silver_clientes.select("id_cliente").withColumn("fl_cliente_cadastrado", F.lit(True))
df_atendentes_referencia = df_silver_atendentes.select("id_atendente").withColumn("fl_atendente_cadastrado", F.lit(True))

# Cruzamento com as dimensões tratadas e com a tabela de horários
df_silver_chamados = (
    df_silver_chamados
    .join(df_motivos_join.select("id_motivo", "motivo_chave"), on="motivo_chave", how="left")
    .join(df_canais_join.select("id_canal", "canal"), on="canal", how="left")
    .join(df_silver_chamados_hora, on="id_chamado", how="left")
    .join(df_clientes_referencia, on="id_cliente", how="left")
    .join(df_atendentes_referencia, on="id_atendente", how="left")
    .withColumn("fl_cliente_cadastrado", F.coalesce(F.col("fl_cliente_cadastrado"), F.lit(False)))
    .withColumn("fl_atendente_cadastrado", F.coalesce(F.col("fl_atendente_cadastrado"), F.lit(False)))
    # Chamado sem registro em Chamados_Hora: as flags de tempo ficam nulas no join e passam a falsas
    .withColumn("fl_espera_valida", F.coalesce(F.col("fl_espera_valida"), F.lit(False)))
    .withColumn("fl_atendimento_valido", F.coalesce(F.col("fl_atendimento_valido"), F.lit(False)))
    .withColumn("fl_ciclo_atendimento_valido", F.coalesce(F.col("fl_ciclo_atendimento_valido"), F.lit(False)))
    .withColumn("fl_cliente_horario_consistente", F.col("id_cliente") == F.col("id_cliente_horario"))
    .withColumn("fl_tempo_espera_consistente", F.col("tempo_espera_informado_segundos") == F.col("tempo_espera_calculado_segundos"))
    .withColumn("fl_tempo_atendimento_consistente", F.col("tempo_atendimento_informado_segundos") == F.col("tempo_atendimento_calculado_segundos"))
    .drop("motivo_chave")
)

# Validações de integridade da Fato (as comparações só valem quando os dois valores existem)
dq_check_unique("silver.tb_chamados", "id_chamado único", df_silver_chamados, ["id_chamado"])
dq_warning("silver.tb_chamados", "motivo identificado", df_silver_chamados, F.col("id_motivo").isNotNull())
dq_warning("silver.tb_chamados", "canal identificado", df_silver_chamados, F.col("id_canal").isNotNull())
dq_warning("silver.tb_chamados", "chamado com horário em Chamados_Hora", df_silver_chamados, F.col("hora_abertura_chamado").isNotNull())
dq_warning("silver.tb_chamados", "cliente encontrado na base de clientes", df_silver_chamados, F.col("fl_cliente_cadastrado"))
dq_warning("silver.tb_chamados", "atendente encontrado na base de atendentes", df_silver_chamados, F.col("fl_atendente_cadastrado"))
dq_warning("silver.tb_chamados", "cliente igual nas duas bases de chamados", df_silver_chamados, F.coalesce(F.col("fl_cliente_horario_consistente"), F.lit(True)))
dq_warning("silver.tb_chamados", "tempo de espera consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("fl_tempo_espera_consistente"), F.lit(True)))
dq_warning("silver.tb_chamados", "tempo de atendimento consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("fl_tempo_atendimento_consistente"), F.lit(True)))

display(df_silver_chamados.limit(10))

## 5. Tratamento de Custos e Pesquisas de Satisfação

Processamento das métricas financeiras e de feedback do cliente.
- **Custos:** Remoção de strings textuais (`R$`, `reais`), tratamento robusto de formatação numérica brasileira (`1.234,56`) e conversão para tipo decimal (`decimal(10,2)`). Valores negativos são preservados para auditoria analítica.
- **Pesquisa de Satisfação:** Padronização das notas de atendimento e validação de intervalo normativo (escala de 1 a 5) em conformidade com as especificações do case.

In [0]:
# Leitura das tabelas bronze de custos e pesquisas
df_bronze_custos = spark.table(f"{bronze_schema}.tb_custos").drop("ingestion_timestamp")
df_bronze_pesquisas = spark.table(f"{bronze_schema}.tb_pesquisa_satisfacao").drop("ingestion_timestamp")

# Limpeza, conversão monetária e remoção de duplicados da base de custos
df_silver_custos = (
    df_bronze_custos
    .filter(F.col("id_custo").isNotNull() & F.col("id_chamado").isNotNull())
    .withColumn("id_chamado", F.trim(F.col("id_chamado")))
    .withColumn("custo", F.lower(F.trim(F.col("custo"))))
    .withColumn("custo", F.trim(F.regexp_replace(F.col("custo"), r"r\$|reais", "")))
    .withColumn("custo",
        F.when(F.col("custo").contains(","), F.regexp_replace(F.regexp_replace(F.col("custo"), r"\.", ""), ",", "."))
        .otherwise(F.col("custo"))
    )
    .withColumn("custo", F.col("custo").try_cast("decimal(10,2)"))
    .dropDuplicates()
)

# Tratamento da pesquisa de satisfação
df_silver_pesquisas = (
    df_bronze_pesquisas
    .filter(F.col("id_pesquisa").isNotNull() & F.col("id_chamado").isNotNull())
    .withColumn("id_chamado", F.trim(F.col("id_chamado")))
    .withColumn("nota_atendimento",
        F.when(F.trim(F.col("nota_atendimento")) == "", None)
        .when(F.upper(F.trim(F.col("nota_atendimento"))) == "NULL", None)
        .otherwise(F.col("nota_atendimento"))
        .try_cast("int")
    )
    .withColumn("nota_valida", F.col("nota_atendimento").between(1, 5))
)

# Validações de qualidade
dq_check_unique("silver.tb_custos", "id_custo único", df_silver_custos, ["id_custo"])
dq_check_unique("silver.tb_pesquisa_satisfacao", "id_pesquisa único", df_silver_pesquisas, ["id_pesquisa"])
dq_warning("silver.tb_custos", "custo convertido", df_silver_custos, F.col("custo").isNotNull())
dq_warning("silver.tb_custos", "custo não negativo", df_silver_custos, F.coalesce(F.col("custo") >= 0, F.lit(True)))
dq_warning("silver.tb_pesquisa_satisfacao", "nota de atendimento entre 1 e 5", df_silver_pesquisas, F.coalesce(F.col("nota_valida"), F.lit(False)))

# Referência dos chamados tratados para validar custos e pesquisas
df_chamados_existentes = df_silver_chamados.select("id_chamado").withColumn("chamado_existe", F.lit(True))

dq_warning("silver.tb_custos", "chamado encontrado em tb_chamados", df_silver_custos.join(df_chamados_existentes, on="id_chamado", how="left"), F.col("chamado_existe"))
dq_warning("silver.tb_custos", "um custo por chamado", df_silver_custos.groupBy("id_chamado").count(), F.col("count") == 1)
dq_warning("silver.tb_pesquisa_satisfacao", "chamado encontrado em tb_chamados", df_silver_pesquisas.join(df_chamados_existentes, on="id_chamado", how="left"), F.col("chamado_existe"))
dq_warning("silver.tb_pesquisa_satisfacao", "uma pesquisa por chamado", df_silver_pesquisas.groupBy("id_chamado").count(), F.col("count") == 1)

display(df_silver_custos.limit(10))
display(df_silver_pesquisas.limit(10))

## 6. Governança de Qualidade, Auditoria e Persistência

Consolidação do processo de validação e persistência das tabelas tratadas na camada Silver.
- **Log de Qualidade (`dq_log`):** Registro histórico em modo `append` segmentado por `run_id`, catalogando métricas de severidade (`BLOCKING`, `WARNING` e `INFO`).
- **Mecanismo de Segurança (Fail-Safe):** Interrupção automática do pipeline antes da gravação em caso de falha em verificações bloqueantes, assegurando a consistência relacional e impedindo a propagação de dados corrompidos para a camada Gold.
- **Persistência Idempotente:** Gravação final em formato Delta utilizando modo `overwrite` para as tabelas de domínio e fatos.

In [0]:
# Tabelas tratadas (Bronze, Silver) a serem validadas e gravadas na camada Silver
tabelas_silver = {
    "tb_atendentes": (df_bronze_atendentes, df_silver_atendentes),
    "tb_motivos": (df_bronze_motivos, df_silver_motivos),
    "tb_canais": (df_bronze_canais, df_silver_canais),
    "tb_clientes": (df_bronze_clientes, df_silver_clientes),
    "tb_chamados_hora": (df_bronze_chamados_hora, df_silver_chamados_hora),
    "tb_chamados": (df_bronze_chamados, df_silver_chamados),
    "tb_custos": (df_bronze_custos, df_silver_custos),
    "tb_pesquisa_satisfacao": (df_bronze_pesquisas, df_silver_pesquisas),
}

# Volume Bronze -> Silver por tabela, para rastrear as linhas descartadas no tratamento
for tabela, (df_bronze, df_silver) in tabelas_silver.items():
    linhas_bronze = df_bronze.count()
    descartadas = linhas_bronze - df_silver.count()
    register_dq(f"silver.{tabela}", "linhas descartadas Bronze -> Silver", "INFO", linhas_bronze, descartadas)
    print(f"[INFO] silver.{tabela} | {descartadas} de {linhas_bronze} linhas descartadas")

# Registro do histórico de qualidade na camada Silver (append, identificado por run_id)
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.dq_log")

display(df_dq_log.orderBy("passed", "table_name"))

# Interrompe o pipeline, antes de gravar a Silver, se alguma validação bloqueante falhou
falhas = [r for r in dq_results if r.severity == "BLOCKING" and not r.passed]
assert not falhas, "Silver não atualizada, falhas bloqueantes: " + "; ".join(f"{r.table_name} | {r.check_name}" for r in falhas)

# Persistência das tabelas tratadas na camada Silver em formato Delta
for tabela, (_, df_silver) in tabelas_silver.items():
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{tabela}")
    print(f"[OK] {silver_schema}.{tabela} gravada")